In [ ]:
# Optional: install required libraries for this lecture
%pip install -q openai pillow


### Step 1: Enter your OpenRouter API key

This notebook uses **OpenRouter** through its OpenAI-compatible API.
The API key is entered interactively and is hidden while you type.


In [ ]:
import getpass
import base64
import mimetypes
from openai import OpenAI

OPENROUTER_API_KEY = getpass.getpass("Enter your OpenRouter API key: ")

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

print("OpenRouter client initialized:", bool(OPENROUTER_API_KEY))


### Step 2: Pick an image

Put a small JPG/PNG in the same folder as this notebook, for example `sample.png`.
The image is converted to a data URL and sent to the multimodal model through OpenRouter.


In [ ]:
image_path = "sample.png"  # change if you use another file name

mime_type, _ = mimetypes.guess_type(image_path)
if mime_type is None:
    mime_type = "image/jpeg"

with open(image_path, "rb") as f:
    image_bytes = f.read()

image_data_url = (
    f"data:{mime_type};base64,"
    + base64.b64encode(image_bytes).decode("utf-8")
)

print(f"Loaded image: {image_path} ({mime_type})")


### Step 3: Describe the image

We'll use a vision-capable model available through OpenRouter.


In [ ]:
result = client.chat.completions.create(
    model="google/gemini-2.5-flash",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe this image in one short sentence."},
                {
                    "type": "image_url",
                    "image_url": {"url": image_data_url}
                },
            ],
        }
    ],
)

print(result.choices[0].message.content)


### Step 4: Ask for specific details

Try a focused instruction such as: "List two colors visible" or "What is the main object?"


In [ ]:
result = client.chat.completions.create(
    model="google/gemini-2.5-flash",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "What is the most expensive article on the menu?"},
                {
                    "type": "image_url",
                    "image_url": {"url": image_data_url}
                },
            ],
        }
    ],
    temperature=0.2,
)

print(result.choices[0].message.content)


### Step 5: OCR (extract text) + structured output

Ask the model to extract readable text and return it as JSON.


In [ ]:
ocr = client.chat.completions.create(
    model="google/gemini-2.5-flash",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Extract any text you can find in this image. Return only a JSON object."
                },
                {
                    "type": "image_url",
                    "image_url": {"url": image_data_url}
                },
            ],
        }
    ],
    temperature=0,
    response_format={"type": "json_object"},
)

print(ocr.choices[0].message.content)


### Other multimodal tasks you can demo

- Quick captioning: "Describe this image in one sentence."
- Object focus: "What is the main object?"
- Counting: "How many people/objects are visible?"
- Color/theme: "List two dominant colors."
- UI reading: "Summarize this screenshot in 20 words."
- Visual Q&A: "Is there any text mentioning 'SALE'?"
- Safety scan: "Is there sensitive info (PII) in this image?"
- Document snippet: "Read the title and author only."

Keep prompts very short and the outputs under ~30–40 words for on-cam pacing.
